# Task 1: Within-Subject EEG Classification

This notebook provides a starter scaffold for Task 1.

In the student release, Task 1 follows a Kaggle-style setup:
- `train.npz` is labeled
- `test.npz` is unlabeled
- you must build your own local validation protocol from the training set

What you must do:
- implement preprocessing
- run the experiment for all 4 required frequency bands
- compare local validation performance across bands
- choose one final approach and export predictions for the official test set


## Important Notes

- Do not use the official test set to tune your model.
- Do not fit preprocessing statistics on the official test set.
- Use the training set to create your own train/validation split.
- Run this notebook from the current Task 1 directory (`part1/task1`) so relative data paths resolve correctly.
- The final CSV must use header `id,label`.

## 1. Data Loading

Load the released within-subject train/test split. The released `.npz` files include
`sfreq` and `channels` metadata, which we propagate into the pipeline so filter cutoffs
are computed correctly without hard-coding the sampling rate.


In [ ]:
from pathlib import Path
import csv
import random

import numpy as np
import pandas as pd

from scipy.signal import butter, sosfiltfilt
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path('data')
TRAIN_FILE = DATA_DIR / 'train.npz'
TEST_FILE = DATA_DIR / 'test.npz'
if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(
        'Task 1 data files not found. Expected data/train.npz and data/test.npz under part1/task1.'
    )

REQUIRED_BANDS = {
    'alpha_mu_8_13':   (8.0, 13.0),
    'beta_13_30':      (13.0, 30.0),
    'broad_4_40':      (4.0, 40.0),
    'high_gamma_70_125': (70.0, 125.0),
}

train_npz = np.load(TRAIN_FILE, allow_pickle=True)
test_npz = np.load(TEST_FILE, allow_pickle=True)

x_train = np.asarray(train_npz['x'], dtype=np.float64)
y_train = np.asarray(train_npz['y'], dtype=np.int64)
x_test = np.asarray(test_npz['x'], dtype=np.float64)
test_ids = (
    np.asarray(test_npz['id'], dtype=np.int64)
    if 'id' in test_npz.files
    else np.arange(len(x_test), dtype=np.int64)
)

# Sampling frequency may live in the npz; fall back to 250 Hz if missing.
def _read_scalar(arr_dict, key, default):
    if key not in arr_dict.files:
        return default
    val = arr_dict[key]
    return float(np.asarray(val).reshape(-1)[0])

sfreq = _read_scalar(train_npz, 'sfreq', 250.0)
n_channels = x_train.shape[1]
n_time = x_train.shape[2]

print(f"x_train: {x_train.shape}  y_train: {y_train.shape}  sfreq={sfreq} Hz")
print(f"x_test : {x_test.shape}  test_ids: {test_ids.shape}")
print(f"channels: {n_channels}, time samples: {n_time}")
unique, counts = np.unique(y_train, return_counts=True)
print(f"label distribution: {dict(zip(unique.tolist(), counts.tolist()))}")


## 2. Preprocessing Designs

Two preprocessing pipelines, both ending in channel-wise log-variance features and a
StandardScaler that is fit only on the training fold.

- **Design A (bandpass-only):** Butterworth bandpass (`sosfiltfilt`) → log-variance per channel.
- **Design B (per-trial z-score + bandpass):** subtract each trial's per-channel mean and
  divide by its per-channel std *before* filtering, then bandpass → log-variance.
  This removes per-trial amplitude/offset variation that can dominate the variance feature.

Both designs share the same classifier head (LinearSVC with `class_weight='balanced'`,
`C=1.0`) so any difference in performance is attributable to the preprocessing.


In [ ]:
def effective_band(low, high, sfreq):
    """Return (low, high) cutoffs that are valid for scipy butter design.

    scipy.signal.butter requires cutoffs strictly below the Nyquist frequency.
    The required 70-125 Hz band sits exactly at Nyquist when sfreq=250 Hz, so we
    clamp the upper cutoff a hair below to keep the filter design valid. The
    nominal band (e.g. "70-125 Hz") is preserved everywhere user-facing; only the
    actual cutoff used internally by the filter changes.
    """
    nyq = sfreq / 2.0
    safe_high = min(high, nyq * 0.999)
    return float(low), float(safe_high)


def design_bandpass(low, high, sfreq, order=4):
    nyq = sfreq / 2.0
    eff_low, eff_high = effective_band(low, high, sfreq)
    wn = (eff_low / nyq, eff_high / nyq)
    if not (0.0 < wn[0] < wn[1] < 1.0):
        raise ValueError(
            f'Band {low}-{high} Hz invalid for sfreq={sfreq} Hz '
            f'(effective cutoffs {eff_low}-{eff_high} -> normalized {wn}).'
        )
    return butter(order, wn, btype='bandpass', output='sos')


def per_trial_zscore(x):
    """Per-trial channel-wise z-score on the raw signal. x: (N, C, T)."""
    mu = x.mean(axis=-1, keepdims=True)
    sigma = x.std(axis=-1, keepdims=True)
    sigma = np.where(sigma < 1e-10, 1.0, sigma)
    return (x - mu) / sigma


def bandpass(x, sos):
    return sosfiltfilt(sos, x, axis=-1)


def log_variance(x):
    var = np.maximum(np.var(x, axis=-1), 1e-10)
    return np.log(var)


def features_design_a(x_train_raw, x_eval_raw, band, sfreq):
    """Design A: bandpass + log-variance."""
    sos = design_bandpass(*band, sfreq=sfreq)
    f_tr = log_variance(bandpass(x_train_raw, sos))
    f_ev = log_variance(bandpass(x_eval_raw, sos))
    return f_tr, f_ev


def features_design_b(x_train_raw, x_eval_raw, band, sfreq):
    """Design B: per-trial z-score -> bandpass -> log-variance."""
    sos = design_bandpass(*band, sfreq=sfreq)
    f_tr = log_variance(bandpass(per_trial_zscore(x_train_raw), sos))
    f_ev = log_variance(bandpass(per_trial_zscore(x_eval_raw), sos))
    return f_tr, f_ev


PREP_DESIGNS = {
    'A_bandpass_only': features_design_a,
    'B_zscore_bandpass': features_design_b,
}


def build_classifier():
    return LinearSVC(C=1.0, class_weight='balanced', random_state=SEED, max_iter=10000)


## 3. Band × Design Comparison via Stratified K-Fold

With only 16 training trials we cannot afford a deep model or a large held-out set.
Stratified K-Fold gives every class representation in each validation fold and uses
all data for both training and evaluation across folds.

`n_splits` is set to `min(4, smallest_class_count)` so each fold gets at least one
sample of every class. Macro-F1 is the primary metric; accuracy is reported alongside.


In [ ]:
min_class_count = int(np.min(np.unique(y_train, return_counts=True)[1]))
n_splits = max(2, min(4, min_class_count))
print(f'Using StratifiedKFold(n_splits={n_splits}); smallest class has {min_class_count} samples.')

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)


def evaluate(design_name, design_fn, band_name, band):
    fold_f1 = []
    fold_acc = []
    for tr_idx, va_idx in skf.split(x_train, y_train):
        x_tr_raw, x_va_raw = x_train[tr_idx], x_train[va_idx]
        y_tr, y_va = y_train[tr_idx], y_train[va_idx]

        f_tr, f_va = design_fn(x_tr_raw, x_va_raw, band, sfreq)
        scaler = StandardScaler().fit(f_tr)
        f_tr_s = scaler.transform(f_tr)
        f_va_s = scaler.transform(f_va)

        clf = build_classifier()
        clf.fit(f_tr_s, y_tr)
        y_pred = clf.predict(f_va_s)
        fold_f1.append(f1_score(y_va, y_pred, average='macro'))
        fold_acc.append(accuracy_score(y_va, y_pred))
    return fold_f1, fold_acc


records = []
for design_name, design_fn in PREP_DESIGNS.items():
    for band_name, band in REQUIRED_BANDS.items():
        eff_low, eff_high = effective_band(band[0], band[1], sfreq)
        clamped = (eff_high != band[1])
        f1s, accs = evaluate(design_name, design_fn, band_name, band)
        records.append({
            'design': design_name,
            'band': band_name,
            'low_hz': band[0],
            'high_hz': band[1],
            'actual_high_hz': eff_high,
            'clamped_to_nyquist': clamped,
            'macro_f1_mean': float(np.mean(f1s)),
            'macro_f1_std':  float(np.std(f1s)),
            'accuracy_mean': float(np.mean(accs)),
            'accuracy_std':  float(np.std(accs)),
            'fold_f1':  [round(s, 4) for s in f1s],
            'fold_acc': [round(s, 4) for s in accs],
        })
        clamp_note = f' (high cutoff clamped to {eff_high:g} Hz for filter validity)' if clamped else ''
        print(
            f"{design_name:22s} | {band_name:18s} "
            f"| macro_f1={np.mean(f1s):.4f}±{np.std(f1s):.4f} "
            f"| acc={np.mean(accs):.4f}±{np.std(accs):.4f}{clamp_note}"
        )

results_df = pd.DataFrame(records).sort_values('macro_f1_mean', ascending=False).reset_index(drop=True)
print('\n=== Sorted results (best first) ===')
print(results_df.drop(columns=['fold_f1', 'fold_acc']).to_string(index=False))


## 4. Best Model Selection

Pick the (design, band) pair with the highest mean Macro-F1, refit on **all** training
data with the same pipeline (StandardScaler fit only on full training set), and report
the on-train fit as a sanity check. Note: the 1.00 train fit on a 16-trial set is
expected and is *not* a measure of generalization — the K-Fold CV above is the only
honest signal.


In [ ]:
best_row = results_df.iloc[0]
best_design = best_row['design']
best_band = best_row['band']
best_band_range = REQUIRED_BANDS[best_band]
best_design_fn = PREP_DESIGNS[best_design]

print(f"Best CV configuration: design={best_design}  band={best_band} ({best_band_range[0]}-{best_band_range[1]} Hz)")
print(f"  CV macro_f1 = {best_row['macro_f1_mean']:.4f} ± {best_row['macro_f1_std']:.4f}")
print(f"  CV accuracy = {best_row['accuracy_mean']:.4f} ± {best_row['accuracy_std']:.4f}")

f_train_full, f_test = best_design_fn(x_train, x_test, best_band_range, sfreq)
final_scaler = StandardScaler().fit(f_train_full)
f_train_full_s = final_scaler.transform(f_train_full)
f_test_s = final_scaler.transform(f_test)

final_clf = build_classifier()
final_clf.fit(f_train_full_s, y_train)

train_pred = final_clf.predict(f_train_full_s)
print(f"\nFinal model train fit (sanity): macro_f1={f1_score(y_train, train_pred, average='macro'):.4f} "
      f"accuracy={accuracy_score(y_train, train_pred):.4f}")


## 5. Submission Generation

Predict the official test set with the chosen pipeline and write a Kaggle-format CSV.


In [ ]:
SUBMISSION_PATH = Path('task1_submission.csv')
test_pred = final_clf.predict(f_test_s).astype(int)

with SUBMISSION_PATH.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label'])
    for sample_id, label in zip(test_ids.tolist(), test_pred.tolist()):
        writer.writerow([int(sample_id), int(label)])

print(f"Wrote {len(test_pred)} predictions to {SUBMISSION_PATH}")
print('Prediction distribution:', dict(zip(*np.unique(test_pred, return_counts=True))))


## 6. Discussion Notes for Report

Use the printed `results_df` table as the source for the report's Task 1 results section.
Talking points the table directly supports:

1. **Frequency-band comparison.** Compare mean Macro-F1 across the four required bands
   (8–13, 13–30, 4–40, 70–125 Hz). For motor-execution EEG the alpha/mu and beta bands
   typically carry the most class-discriminative information; the 70–125 Hz band is
   often noisier on consumer-grade EEG and may underperform.
2. **Preprocessing comparison.** Design A (bandpass only) is the simplest pipeline.
   Design B adds a per-trial z-score before filtering, which removes per-trial offset
   and amplitude drift before computing variance features. State which design has the
   higher mean Macro-F1 and explain in terms of what the extra normalization removes.
3. **Validation honesty.** With only 16 trials, single-split val accuracy is unstable;
   StratifiedKFold averages over multiple folds for a more reliable signal. Report
   mean ± std, not just the best fold.
4. **Limitations.** 16 trials × 4 classes is a very small training set. Differences
   in mean Macro-F1 of < 1 std should be considered noise.
